# Task 2: Product category clustering

Goal: group product reviews into 4-6 meaningful meta-categories using review text and product characteristics. Run all cells from the repository root or `models` directory after installing `requirements-train.txt`. The first run downloads a pinned sentence embedding model; text embeddings are cached in the system temporary directory so weight experiments do not require repeated inference.

One row represents a distinct `(id, name, brand, categories)` profile, with averaged review embeddings. Reviews inherit their profile's cluster. This avoids merging unrelated review themes solely because the source metadata is inconsistent.

**Kernel setup:** the first code cell installs `sentence-transformers` into the selected kernel if missing. For this workspace, `.venv-2/Scripts/python.exe` already has the dependencies installed. In VS Code, select that interpreter with the notebook's kernel picker and choose **Run All**. If installation asks for a kernel restart, restart it and run all cells again.


Final output: six learned clusters are consolidated into five meta-categories, with clusters 3 and 4 sharing the label **Tablets and e-readers**.

In [1]:
# Install into the selected notebook kernel, rather than another environment.
import importlib.util
import sys

print(f'Notebook Python: {sys.executable}')
if importlib.util.find_spec('sentence_transformers') is None:
    get_ipython().run_line_magic('pip', 'install "sentence-transformers>=3,<7"')


Notebook Python: c:\Users\zinam\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/1429_1.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/1429_1.csv")

df = pd.read_csv(DATA_PATH, low_memory=False)
display(df.head())

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [3]:
category_counts = (
    df["categories"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

display(category_counts)

categories
Electronics            42291
Computers & Tablets    21719
Tablets                21383
All Tablets            18413
iPad & Tablets         17784
                       ...  
Rice Dishes                1
Ready Meals                1
Beauty                     1
Moisturizers               1
Lotions                    1
Name: count, Length: 151, dtype: int64

In [4]:
df["id"].nunique()

42

## Goal and approach

The source contains inconsistent names and category paths. Compare the original category/product/review scales **4:2:1** with five alternatives, using KMeans for K=4, 5, and 6. Ratings and recommendation fields are excluded.

For each configuration, report silhouette in its own combined feature space and in the **same review-embedding space**. Only the latter is directly comparable across feature weights. Also measure agreement across random seeds using adjusted Rand index (ARI). Inspect keywords, product profiles, and ten review examples per cluster before choosing descriptive names.

The selected configuration is **0.5:1:2 with K=6**. It gives more influence to review content while retaining product characteristics, and separates the previous broad group into recognizable categories. Selection is exploratory on this dataset, not a held-out accuracy estimate.

In [5]:
print(f"Rows: {len(df):,}; columns: {df.shape[1]}")
print(f"Products with names: {df['name'].nunique(dropna=True):,}")
print(f"Unique product IDs: {df['id'].nunique(dropna=True):,}")
display(df[['name', 'brand', 'categories', 'reviews.title', 'reviews.text']].isna().sum().rename('missing').to_frame())

display(df['name'].fillna('[name missing]').value_counts().head(10).rename_axis('product').to_frame('review_count'))

Rows: 34,660; columns: 21
Products with names: 48
Unique product IDs: 42


,missing
name,6760
brand,0
categories,0
reviews.title,6
reviews.text,1


,review_count
product,
"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta",10966
[name missing],6760
"Echo (White),,,\r\nEcho (White),,,",3309
"Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrome Paperwhite - touchscreen - Wi-Fi - black,,,",3176
"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",2814
"Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,",2527
"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case",1685
"Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,,",1038
"Kindle Voyage E-reader, 6 High-Resolution Display (300 ppi) with Adaptive Built-in Light, PagePress Sensors, Wi-Fi - Includes Special Offers,",580


## Build product and review features

Embed category paths, product names/brands, and review title/body text with the same sentence model. Average review embeddings per metadata profile and normalize each block. The comparison below varies the block scales without changing these embeddings. Because the blocks are concatenated, their squared-distance contributions scale with the **square** of each weight; the selected 0.5:1:2 scales give 0.25:1:4 contributions.

In [6]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

product_columns = ['id', 'name', 'brand', 'categories']
product_df = df[product_columns].drop_duplicates().reset_index(drop=True)

def unique_category_labels(value):
    seen = set()
    terms = []
    for term in str(value).split(','):
        term = term.strip()
        normalized = term.casefold()
        if term and normalized not in seen:
            seen.add(normalized)
            terms.append(term)
    return terms

category_text = product_df['categories'].fillna('').map(
    lambda value: ' '.join(unique_category_labels(value))
)
product_text = product_df[['name', 'brand']].fillna('').agg(' '.join, axis=1)
import hashlib
import json
import tempfile

MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'
MODEL_REVISION = '1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
embedding_model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION)
CACHE_DIR = Path(tempfile.gettempdir()) / 'customer-review-embedding-cache'
CACHE_DIR.mkdir(exist_ok=True)


def encode_texts(texts):
    payload = json.dumps([MODEL_ID, MODEL_REVISION, texts], ensure_ascii=False)
    key = hashlib.sha256(payload.encode('utf-8')).hexdigest()
    cache_path = CACHE_DIR / f'{key}.npy'
    if cache_path.exists():
        embeddings = np.load(cache_path, allow_pickle=False)
    else:
        embeddings = embedding_model.encode(
            texts, batch_size=64, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        )
        np.save(cache_path, embeddings, allow_pickle=False)
    assert embeddings.shape == (len(texts), 384), 'Unexpected cached embedding shape.'
    return embeddings

category_matrix = encode_texts(category_text.tolist())
product_matrix = encode_texts(product_text.tolist())

review_rows = df[product_columns].copy()
review_rows['review_text'] = (
    df[['reviews.title', 'reviews.text']].fillna('').agg(' '.join, axis=1).str.strip()
)
review_rows = review_rows[review_rows['review_text'].ne('')].merge(
    product_df[product_columns].assign(_product_index=np.arange(len(product_df))),
    on=product_columns, how='left', validate='many_to_one',
)
if review_rows.empty:
    raise ValueError('No non-empty review text is available for embedding.')

review_embeddings = encode_texts(review_rows['review_text'].tolist())
review_indices = review_rows['_product_index'].to_numpy(dtype=np.int64)
review_sums = np.zeros((len(product_df), review_embeddings.shape[1]), dtype=np.float32)
np.add.at(review_sums, review_indices, review_embeddings)
review_counts = np.bincount(review_indices, minlength=len(product_df))
review_matrix = normalize(review_sums / np.maximum(review_counts, 1)[:, None])

print(f'Unique product records: {len(product_df):,}')
print(f"Embedding features per text block: {category_matrix.shape[1]:,}")
print(f'Embedded review rows: {len(review_rows):,}')
print(f"Features per combined profile: {category_matrix.shape[1] * 3:,}")

W1006 16:47:19.148000 29256 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Unique product records: 89
Embedding features per text block: 384
Embedded review rows: 34,660
Features per combined profile: 1,152


## Compare feature weights and 4-6 clusters

Compare six scale combinations and three values of K, keeping the same embeddings. The fixed review-space silhouette measures how well each grouping separates review content. The fitted-space silhouette measures separation in that configuration's weighted space and should not be used alone to rank configurations with different weights. Seed stability is the mean ARI against three other fits; it measures reproducibility, not semantic correctness.

In [7]:
from sklearn.metrics import adjusted_rand_score

WEIGHT_OPTIONS = [(4, 2, 1), (2, 2, 1), (1, 1, 1),
                  (1, 1, 2), (0.5, 1, 2), (0.5, 0.5, 2)]
candidate_results = []
candidate_models = {}
for weights in WEIGHT_OPTIONS:
    features = normalize(np.hstack([
        category_matrix * weights[0], product_matrix * weights[1],
        review_matrix * weights[2],
    ]))
    for k in (4, 5, 6):
        candidate = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = candidate.fit_predict(features)
        seed_agreement = [
            adjusted_rand_score(labels, KMeans(
                n_clusters=k, random_state=seed, n_init=20,
            ).fit_predict(features))
            for seed in (7, 21, 84)
        ]
        candidate_models[(weights, k)] = {'estimator': candidate, 'labels': labels}
        counts = np.bincount(labels)
        candidate_results.append({
            'weights': weights, 'k': k,
            'silhouette_fitted_space': silhouette_score(features, labels, metric='cosine'),
            'silhouette_fixed_review_space': silhouette_score(review_matrix, labels, metric='cosine'),
            'mean_seed_ari': np.mean(seed_agreement),
            'smallest_profile_cluster': int(counts.min()),
            'largest_profile_cluster': int(counts.max()),
        })
comparison = pd.DataFrame(candidate_results).sort_values(
    'silhouette_fixed_review_space', ascending=False,
).reset_index(drop=True)
display(comparison.round(3))


,weights,k,silhouette_fitted_space,silhouette_fixed_review_space,mean_seed_ari,smallest_profile_cluster,largest_profile_cluster
0,"(0.5, 1, 2)",6,0.301,0.381,0.903,5,26
1,"(0.5, 0.5, 2)",6,0.361,0.377,0.915,5,27
2,"(1, 1, 2)",6,0.336,0.373,0.870,4,28
3,"(0.5, 0.5, 2)",5,0.329,0.345,0.913,7,26
4,"(0.5, 1, 2)",5,0.280,0.344,0.827,7,30
5,"(0.5, 0.5, 2)",4,0.333,0.343,0.878,7,32
6,"(1, 1, 2)",5,0.309,0.333,0.837,7,31
7,"(1, 1, 1)",5,0.286,0.313,0.811,5,25
8,"(0.5, 1, 2)",4,0.233,0.296,0.641,15,28
9,"(1, 1, 1)",4,0.268,0.282,0.991,16,29


In [8]:
# Highest fixed review-space silhouette; checked against keywords and samples.
SELECTED_WEIGHTS = (0.5, 1, 2)
N_CLUSTERS = 6
X = normalize(np.hstack([
    category_matrix * SELECTED_WEIGHTS[0], product_matrix * SELECTED_WEIGHTS[1],
    review_matrix * SELECTED_WEIGHTS[2],
]))
# Fit directly so this cell does not depend on running the comparison sweep.
model = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
product_df = product_df.drop(columns=['learned_cluster_id', 'meta_category'], errors='ignore')
product_df['cluster_id'] = model.fit_predict(X)
df = df.drop(columns=['cluster_id', 'learned_cluster_id', 'meta_category'], errors='ignore').merge(
    product_df[product_columns + ['cluster_id']],
    on=product_columns, how='left', validate='many_to_one',
)

centers = np.vstack([
    np.asarray(X[product_df['cluster_id'].to_numpy() == cluster_id].mean(axis=0)).ravel()
    for cluster_id in range(N_CLUSTERS)
])
centers = normalize(centers)
cluster_rows = []

for cluster_id in range(N_CLUSTERS):
    cluster_mask = product_df['cluster_id'].to_numpy() == cluster_id
    members = product_df.loc[cluster_mask]
    products = members['name'].dropna().value_counts().head(5)
    category_counts = (
        members['categories'].dropna().map(unique_category_labels).explode()
        .value_counts().head(8)
    )
    member_positions = np.flatnonzero(cluster_mask)
    similarities = X[member_positions] @ centers[cluster_id]
    representative_positions = member_positions[np.argsort(similarities)[-3:][::-1]]
    representative_products = product_df.iloc[representative_positions]['name'].fillna('[missing name]')
    cluster_rows.append({
        'cluster_id': cluster_id,
        'product_count': len(members),
        'top_products': ' | '.join(products.index.astype(str)),
        'representative_products': ' | '.join(representative_products.astype(str)),
        'common_categories': ', '.join(category_counts.index.astype(str)),
    })

cluster_summary = pd.DataFrame(cluster_rows).set_index('cluster_id')
with pd.option_context('display.max_colwidth', None, 'display.width', None):
    display(cluster_summary)

# Keyword analysis explains the groups; it is not used to fit or label the model.
# Each profile contributes one normalized TF-IDF document regardless of review count.
from sklearn.feature_extraction.text import TfidfVectorizer
profile_texts = review_rows.groupby('_product_index')['review_text'].agg(' '.join)
profile_texts = profile_texts.reindex(range(len(product_df)), fill_value='')
keyword_vectorizer = TfidfVectorizer(
    stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.85, max_features=8000,
)
keyword_matrix = keyword_vectorizer.fit_transform(profile_texts)
terms = keyword_vectorizer.get_feature_names_out()
keyword_rows = []
for cluster_id in range(N_CLUSTERS):
    mask = product_df['cluster_id'].to_numpy() == cluster_id
    scores = np.asarray(keyword_matrix[mask].mean(axis=0)).ravel()
    keyword_rows.append({
        'cluster_id': cluster_id,
        'top_review_keywords': ', '.join(terms[np.argsort(scores)[-12:][::-1]]),
    })
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(keyword_rows).set_index('cluster_id'))


,product_count,top_products,representative_products,common_categories
cluster_id,,,,
0,7,"Amazon Kindle Lighted Leather Cover,,,\r\nAmazon Kindle Lighted Leather Cover,,, | Amazon Kindle Lighted Leather Cover,,,\r\nKindle Keyboard,,, | Kindle Keyboard,,,\r\nKindle Keyboard,,, | Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,, | Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case","Amazon Kindle Lighted Leather Cover,,,\r\nAmazon Kindle Lighted Leather Cover,,, | Kindle Keyboard,,,\r\nKindle Keyboard,,, | Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case","Kindle Store, Amazon Device Accessories, Covers, Kindle E-Reader Accessories, Electronics, eBook Readers & Accessories, Kindle (5th Generation) Accessories, Kindle (5th Generation) Covers"
1,5,"Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black,,,\r\nFire HD 8 Tablet with Alexa, 8 HD Display, 16 GB, Tangerine - with Special Offers"", | Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,, | Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black | Echo (White),,,\r\nEcho (White),,,","Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black | Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,, | [missing name]","Electronics, Categories, Streaming Media Players, Amazon Devices, Fire TV, Kindle Store, Fire Tablets, Tablets"
2,8,"Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,\r\nAmazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders,,, | All-New Kindle E-reader - Black, 6 Glare-Free Touchscreen Display, Wi-Fi - Includes Special Offers,, | Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\r\nAmazon Kindle Fire Hd (3rd Generation) 8gb,,,",[missing name] | [missing name] | [missing name],"Power Adapters & Cables, Amazon Device Accessories, Kindle Store, Amazon Devices & Accessories, Kindle E-Reader Accessories, Kindle Paperwhite Accessories, Fire Tablet Accessories, Electronics"
3,21,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black | Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,, | Kindle Voyage E-reader, 6 High-Resolution Display (300 ppi) with Adaptive Built-in Light, PagePress Sensors, Wi-Fi - Includes Special Offers, | Kindle Oasis E-reader with Leather Charging Cover - Merlot, 6 High-Resolution Display (300 ppi), Wi-Fi - Includes Special Offers,, | Kindle Keyboard,,,\r\nKindle Keyboard,,,","Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrome Paperwhite - touchscreen - Wi-Fi - black,,, | Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black | Kindle Voyage E-reader, 6 High-Resolution Display (300 ppi) with Adaptive Built-in Light, PagePress Sensors, Wi-Fi - Includes Special Offers,","Computers & Tablets, Kindle E-readers, E-Readers & Accessories, Electronics, eBook Readers, iPad & Tablets, Electronics Features, All Tablets"
4,26,"Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,, | Echo (White),,,\r\nEcho (White),,, | Fire HD 8 Tablet with Alexa, 8 HD Display, 32 GB, Tangerine - with Special Offers, | Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta | Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta | Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta | Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black","Tablets, Computers & Tablets, All Tablets, Fire Tablets, Electronics, iPad & Tablets, Frys, Computers/Tablets & Networking"
5,22,"Certified Refurbished Amazon Fire TV Stick (Previous Generation - 1st),,,\r\nCertified Refurbished Amazon Fire TV Stick (Previous Generation - 1st),,, | Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black | Echo (White),,,\r\nEcho (White),,, | Echo (White),,,\r\nF

,top_review_keywords
cluster_id,
0,"cover, kindle, case, leather, light, color, stand, kindle cover, does, hinge, like, sturdy"
1,"tv, prime, remote, amazon, roku, box, stick, apps, apple tv, ads, apple, br"
2,"charger, kindle, stars, works, great, usb, charging, charge, cord, cable, does, just"
3,"kindle, great, reading, reader, voyage, love, read, light, books, easy, paperwhite, page"
4,"tablet, great, kindle, kids, great tablet, good, love, use, easy, price, books, bought"
5,"echo, alexa, music, great, home, fun, love, amazon, speaker, use, product, amazon echo"


## Inspect product and review examples by cluster

Inspect unique product metadata and sampled review text for each cluster. Review rows receive their product's cluster assignment after the product-level clustering. If a cluster mixes unrelated products or review topics, revisit the feature weights or compare another value of `k` before assigning names.

In [9]:
EXAMPLES_PER_CLUSTER = 10
example_rows = []

for cluster_id, members in product_df.groupby('cluster_id', sort=True):
    example_rows.append(
        members.sample(
            n=min(EXAMPLES_PER_CLUSTER, len(members)),
            random_state=42,
        )
    )

cluster_examples = pd.concat(example_rows)
cluster_examples = cluster_examples[
    ['cluster_id', 'name', 'brand', 'categories']
].sort_values('cluster_id').reset_index(drop=True)
with pd.option_context('display.max_colwidth', None):
    display(cluster_examples)

,cluster_id,name,brand,categories
0,0,"Amazon Kindle Lighted Leather Cover,,,\r\nAmazon Kindle Lighted Leather Cover,,,",Amazon,"Electronics,eBook Readers & Accessories,Covers,Kindle Store,Amazon Device Accessories,Kindle E-Reader Accessories,Kindle (5th Generation) Accessories,Kindle (5th Generation) Covers"
1,0,"Amazon Kindle Lighted Leather Cover,,,\r\nKindle Keyboard,,,",Amazon,"Electronics,eBook Readers & Accessories,Covers,Kindle Store,Amazon Device Accessories,Kindle E-Reader Accessories,Kindle (5th Generation) Accessories,Kindle (5th Generation) Covers"
2,0,NaN,Amazon,"Cases,Kindle Store,Amazon Device Accessories,Accessories,Tablet Accessories"
3,0,"Kindle Keyboard,,,\r\nKindle Keyboard,,,",Amazon,"Electronics,eBook Readers & Accessories,Covers,Kindle Store,Amazon Device Accessories,Kindle E-Reader Accessories,Kindle (5th Generation) Accessories,Kindle (5th Generation) Covers"
4,0,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case",Amazon,"Amazon Device Accessories,Kindle Store,Kindle Touch (4th Generation) Accessories,Kindle E-Reader Accessories,Covers,Kindle Touch (4th Generation) Covers"
5,0,"Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,,",Amazon,"Frys,Software & Books,eReaders & Accessories,Tablet Cases Covers,Tablet Accessories,Computer Accessories"
6,0,NaN,Amazon Digital Services Inc.,"Electronics,Amazon Device Accessories,Kindle Store,Covers,Kindle E-Reader Accessories,Kindle DX (2nd Generation, Global Wireless) Accessories"
7,1,"Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,,",Amazon,"Electronics,Categories,Streaming Media Players,Amazon Devices"
8,1,NaN,Amazon Fire Tv,"Back To College,College Electronics,College Tvs & Home Theater,Electronics,Tvs & Home Theater,Streaming Devices,Featured Brands,Amazon Devices,Holiday Shop,Ways To Shop,TV & Home Theater,Streaming Media Players,All Streaming Media Players,TVs Entertainment,Video Games,Kindle Store,Electronics Features,Kids & Family,Fire TV"
9,1,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Black",Amazon,"Electronics,Categories,Fire TV,Kindle Store"


In [10]:
EXAMPLES_PER_CLUSTER = 10
# Select product profiles first so products with thousands of reviews do not
# dominate inspection. Use one non-empty review per sampled profile, then
# fill any remaining slots from other review rows in the cluster.
review_candidates = review_rows.merge(
    product_df[product_columns + ['cluster_id']],
    on=product_columns, how='left', validate='many_to_one',
)
sampled_reviews = []
for cluster_id, members in review_candidates.groupby('cluster_id', sort=True):
    profile_ids = members['_product_index'].drop_duplicates().sample(
        n=min(EXAMPLES_PER_CLUSTER, members['_product_index'].nunique()),
        random_state=42,
    )
    chosen = pd.concat([
        members[members['_product_index'].eq(profile_id)].sample(n=1, random_state=42)
        for profile_id in profile_ids
    ])
    remaining = members.drop(index=chosen.index)
    extra_count = min(EXAMPLES_PER_CLUSTER - len(chosen), len(remaining))
    if extra_count:
        chosen = pd.concat([chosen, remaining.sample(n=extra_count, random_state=42)])
    sampled_reviews.append(chosen)

review_examples = pd.concat(sampled_reviews)[
    ['cluster_id', 'name', 'review_text']
].sort_values('cluster_id').reset_index(drop=True)
with pd.option_context('display.max_colwidth', None):
    display(review_examples)


,cluster_id,name,review_text
0,0,"Amazon Kindle Lighted Leather Cover,,,\r\nAmazon Kindle Lighted Leather Cover,,,","A Step Down In Quality I owned a Kindle Keyboard for about a year, and I purchased the leather lighted case for that unit. While the corner light left a little to be desired as far as even lighting, overall I was very pleased with the case. Based on that experience, I went ahead and purchased the lighted cover for my new Kindle. I can't say this experience has as been as positive.Let's start off with what's good about the case. The Kindle battery powered light is a big improvement. The light is now centered on the screen, so illumination is more even now. I don't use the light often, but in the cases where I do it will be nice to have more balanced coverage. One positive on the case itself is that I appreciate how much lighter this one feels than the previous generation. The hinge area is smaller, which means less of the spine folding effect the other one took on after heavy use. I never took my Kindle Keyboard out of the case after it was put in, so the snug fit I have seen complaints about is not an issue for me. It IS tight, though, so if you don't plan to keep the case on full time be aware it may take some effort to get it back out. In my situation, that tight fit is exactly what I want, but your mileage may vary.Now on to the negatives. The case does feel very cheap, especially with no change in price point. The previous cover felt like a solidly made piece of work, whereas the new cover only feels half as sturdy. The thinness of the case was no doubt done to minimize its weight, so I realize this is a personal issue. Outside the thickness, though, it just doesn't look or feel as well made. The textured surface of the previous model you that leather feel, very much like a real book. Finally, the most stunning oversight is the lack of a closure device.Read more"
1,0,"Amazon Kindle Lighted Leather Cover,,,\r\nKindle Keyboard,,,","High Quality Case For Your Kindle I purchased this Kindle case in the Saddle Tan color. The leather is high quality (it has that wonderful new smell as well) and has beautiful stitching around the border on the front. The case does not add any bulk to the Kindle. I find it quite comfortable to hold in one hand or two, as the front cover folds behind the case, and I truly feel like I am holding an actual book while reading.At the top of the case, there is a plastic arm that pulls upward to reveal a light, which illuminates the screen of the Kindle perfectly in a dark room. My eyes have no trouble reading with the light activated. The light is powered by the Kindle itself, by way of two metal prongs inside the case making contact with two metal tabs on the lower back of the Kindle. The arm for the light does seem a bit delicate, and I could see a young child inadvertently breaking this if care is not shown. I fear that over time, this will be the first piece to stop functioning as the case is regularly used.My only criticism of this case is that there is no way to secure the front cover from opening. It would have been nice to include a clasp, or some kind of stretchable band to keep the case closed. I paid 54 for this case, which is almost as expensive as the Kindle itself. At that price, the designers should have had the forethought to include a means of securing the case from opening. Aside from this potential problem, this Kindle case is perfect, and adds a bit of sophistication to an already awesome device.Kudos, Amazon!"
2,0,NaN,"Four Stars it's a bit flimsy, but does stand up, so I'm not complaining..."
3,0,"Kindle Keyboard,,,\r\nKindle Keyboard,,,","Best Option Out There I read every single review for this cover before I decided to buy it, but I am glad that I finally decided to pull the trigger. I have never had a Kindle or another cover, so my review is based on a first-timer's perspective. This is my feedback:1) I agree that this cover is expensive. Ridiculously expensive. 

## Interpretation and quality comparison

The dataset has **34,660 reviews**, **42 product IDs**, and **89 metadata profiles**. Profiles are not distinct physical products; one source ID can contain conflicting metadata.

| Measure | Original 4:2:1, K=4 | Selected 0.5:1:2, K=6 |
|---|---:|---:|
| Silhouette in the same review-embedding space | 0.201 | 0.381 |
| Silhouette in each model's own fitted space | 0.411 | 0.301 |
| Mean seed agreement (ARI, seeds 7/21/84 vs 42) | 0.912 | 0.903 |

**Why select this model:** the fixed review-space silhouette improves by about **0.180**, and keywords and samples now identify separate covers, chargers, and streaming groups. The fitted-space score decreases, so this is not a claim that every metric improved. The selected fit is reasonably consistent across seeds, with slightly lower stability than the baseline. K=6 fits the required 4-6 range and retains useful distinctions that merge at smaller K.

| Cluster | Profiles | Descriptive name | Evidence and exceptions |
|---|---:|---|---|
| 0 | 7 | Cases and protective covers | Keywords include cover, case, leather, light, stand; reviews discuss protective covers and their build quality. |
| 1 | 5 | Fire TV and streaming devices | Keywords include TV, prime, remote, Roku, box, stick; reviews discuss streaming services and setup. An ads/apps review is less specific. |
| 2 | 8 | Chargers, cables and power accessories | Keywords include charger, USB, charging, cord, cable; reviews mainly concern charging. A rare food review remains an outlier. |
| 3 + 4 | 47 | Tablets and e-readers (merged) | Combined Kindle reading themes and Fire tablet themes: reading, Paperwhite/Voyage, apps, children, and entertainment. Some metadata is inconsistent. |
| 5 | 22 | Echo speakers and smart-home devices | Reviews discuss Alexa, music, voice commands, and home automation. A power-adapter review discusses use with Echo, so accessories can still overlap. |

**Limits:** this is evidence of improved review-based separation and more interpretable themes on this dataset, not ground-truth category accuracy. The metric uses the same review embeddings included in fitting, so it is not independent validation. Cluster sizes are uneven: the covers group has only 36 reviews across seven profiles, while tablets have 17,600 reviews. Metadata errors, generic reviews, and rare non-electronics remain; forcing a single food review into its own cluster would consume one of the six groups. No reviews are removed or manually reassigned, and ratings are not used. Names interpret the fitted groups; they are not training labels. Recheck names after changing data, weights, K, or model.


### Final consolidation: five output clusters

Clusters 3 and 4 share the final label **Tablets and e-readers** (47 profiles, 21,759 reviews). The export contains five consolidated `cluster_id` values and five meta-categories. Original six-cluster assignments are retained in `learned_cluster_id` for diagnostics. The scores above describe the six learned clusters, not the consolidated five-category output.

In [11]:
# Merge Kindle/e-reader and tablet clusters into one final cluster.
N_META_CATEGORIES = 5
cluster_names = {
    0: 'Cases and protective covers',
    1: 'Fire TV and streaming devices',
    2: 'Chargers, cables and power accessories',
    3: 'Tablets and e-readers',
    4: 'Echo speakers and smart-home devices',
}

# Preserve the learned labels for diagnostics; publish five consolidated IDs.
FINAL_CLUSTER_IDS = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3, 5: 4}
for frame in (product_df, df):
    if 'learned_cluster_id' not in frame:
        frame['learned_cluster_id'] = frame['cluster_id']
    frame['cluster_id'] = frame['learned_cluster_id'].map(FINAL_CLUSTER_IDS)
    frame['meta_category'] = frame['cluster_id'].map(cluster_names)

assert product_df.loc[product_df['learned_cluster_id'].isin([3, 4]), 'cluster_id'].nunique() == 1
assert product_df['cluster_id'].nunique() == N_META_CATEGORIES
if product_df['meta_category'].isna().any() or df['meta_category'].isna().any():
    raise ValueError('Every cluster must have a descriptive name.')

product_category_counts = (
    product_df.groupby('meta_category', sort=False)
    .size().rename('product_profiles')
)
review_category_counts = (
    df.groupby('meta_category', sort=False).size().rename('review_rows')
)
display(
    pd.concat([product_category_counts, review_category_counts], axis=1)
    .sort_values('product_profiles', ascending=False)
)
display(product_df[['cluster_id', 'meta_category', 'name']].sort_values(['cluster_id', 'name']).reset_index(drop=True))
assert df['meta_category'].nunique() == N_META_CATEGORIES


,product_profiles,review_rows
meta_category,,
Tablets and e-readers,47,21759
Echo speakers and smart-home devices,22,7279
"Chargers, cables and power accessories",8,512
Cases and protective covers,7,36
Fire TV and streaming devices,5,5074


,cluster_id,meta_category,name
0,0,Cases and protective covers,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz..."
1,0,Cases and protective covers,"Amazon Kindle Lighted Leather Cover,,,\r\nKind..."
2,0,Cases and protective covers,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...
3,0,Cases and protective covers,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16..."
4,0,Cases and protective covers,"Kindle Keyboard,,,\r\nKindle Keyboard,,,"
...,...,...,...
84,4,Echo speakers and smart-home devices,New Amazon Kindle Fire Hd 9w Powerfast Adapter...
85,4,Echo speakers and smart-home devices,New Amazon Kindle Fire Hd 9w Powerfast Adapter...
86,4,Echo speakers and smart-home devices,NaN
87,4,Echo speakers and smart-home devices,NaN


In [12]:
OUTPUT_PATH = DATA_PATH.parent / 'reviews_with_meta_categories.csv'
source_df = pd.read_csv(DATA_PATH, low_memory=False)
# Verify original values and source-row order, not only the row count.
pd.testing.assert_frame_equal(source_df, df[source_df.columns])
assert df[['cluster_id', 'meta_category']].notna().all().all()
assert 4 <= N_CLUSTERS <= 6
assert product_df['learned_cluster_id'].nunique() == N_CLUSTERS
assert product_df['cluster_id'].nunique() == N_META_CATEGORIES
assert df['meta_category'].nunique() == N_META_CATEGORIES
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(df):,} reviews in original order with {N_META_CATEGORIES} final clusters/meta-categories (from {N_CLUSTERS} learned clusters) to {OUTPUT_PATH}')


Saved 34,660 reviews in original order with 5 final clusters/meta-categories (from 6 learned clusters) to ..\data\reviews_with_meta_categories.csv
